# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 02b - Exploratory data analysis and correlation matrices

This notebook analyzes the validated integrated graph before model development. It produces reproducible tables and saved figures for playlist composition, graph degree distributions, metadata coverage, popularity, genres, and numeric correlations.

Correlation is descriptive: it does not prove causation or recommendation quality. Full-graph degrees and popularity are useful for understanding bias, but they are not silently added to the leakage-safe GNN inputs.

## 1. Environment and analysis inputs

Run notebooks 01, 01b, and 02 before this notebook.

In [ ]:
import os
import sys
import math
from pathlib import Path


def find_project_root():
    required = Path("data/processed/gnn_integrated/playlist_track_edges.parquet")
    for anchor in [Path.cwd().resolve(), Path(sys.executable).resolve()]:
        for parent in [anchor, *anchor.parents]:
            for candidate in [parent, parent / "art_xharra"]:
                if (candidate / required).exists():
                    return candidate.resolve()
    raise FileNotFoundError("Integrated graph outputs are missing. Run notebook 02 first.")


PROJECT_ROOT = find_project_root()
JDK_ROOT = PROJECT_ROOT / ".tools/jdk17/jdk-17.0.20+8"
HADOOP_ROOT = PROJECT_ROOT / ".tools/hadoop"
os.environ["JAVA_HOME"] = str(JDK_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = str(JDK_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")
if os.name == "nt":
    os.environ["HADOOP_HOME"] = str(HADOOP_ROOT)
    os.environ["PATH"] = str(HADOOP_ROOT / "bin") + os.pathsep + os.environ["PATH"]

from pyspark import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

spark = (
    SparkSession.builder.master("local[*]")
    .appName("Integrated-Graph-EDA")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

GRAPH_ROOT = PROJECT_ROOT / "data/processed/gnn_integrated"
PLAYLIST_CLEAN_PATH = PROJECT_ROOT / "data/processed/playlists/spud_playlists_clean.parquet"
SPUD_TRACKS_PATH = PROJECT_ROOT / "data/processed/playlists/spud_tracks_clean.parquet"
CATALOG_TRACKS_PATH = PROJECT_ROOT / "data/processed/tracks_enriched.parquet"
FIGURE_ROOT = PROJECT_ROOT / "reports/figures/data_analysis"
REPORTS_ROOT = PROJECT_ROOT / "reports"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180, "axes.titleweight": "bold"})
print(f"Project root: {PROJECT_ROOT}")
print(f"Figures     : {FIGURE_ROOT.relative_to(PROJECT_ROOT)}")

In [ ]:
playlist_nodes = spark.read.parquet(str(GRAPH_ROOT / "playlist_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
track_nodes = spark.read.parquet(str(GRAPH_ROOT / "track_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
artist_nodes = spark.read.parquet(str(GRAPH_ROOT / "artist_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
playlist_track_edges = spark.read.parquet(str(GRAPH_ROOT / "playlist_track_edges.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
track_artist_edges = spark.read.parquet(str(GRAPH_ROOT / "track_artist_edges.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
playlist_clean = spark.read.parquet(str(PLAYLIST_CLEAN_PATH))
spud_tracks = spark.read.parquet(str(SPUD_TRACKS_PATH))
catalog_tracks = spark.read.parquet(str(CATALOG_TRACKS_PATH))

analysis_counts = {
    "playlist_nodes": playlist_nodes.count(),
    "track_nodes": track_nodes.count(),
    "artist_nodes": artist_nodes.count(),
    "playlist_track_edges": playlist_track_edges.count(),
    "track_artist_edges": track_artist_edges.count(),
}
display(pd.DataFrame(analysis_counts.items(), columns=["dataset", "rows"]))

## 2. Build analysis views

Degrees are derived from the complete graph for descriptive analysis only. The later model must recompute them from training edges after validation/test links have been removed.

In [ ]:
playlist_analysis = (
    playlist_nodes.select(
        "playlist_node_id", "playlist_id", "playlist_title", "track_count",
        "artist_count", "computed_duration_seconds", "playlist_size_band",
        "eligible_for_link_prediction", "audio_matched_track_count", "audio_coverage_ratio",
    )
    .join(playlist_clean.select("playlist_id", "mean_spud_popularity"), "playlist_id", "left")
    .withColumn("title_character_count", F.length("playlist_title").cast("double"))
    .withColumn("log_track_count", F.log1p("track_count"))
    .withColumn("log_duration_seconds", F.log1p("computed_duration_seconds"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

track_degrees = playlist_track_edges.groupBy("dst_track_node_id").count().withColumnRenamed("count", "playlist_degree")
track_analysis = (
    track_nodes.select(
        "track_node_id", "spotify_track_id", "spud_popularity",
        "audio_features_available", "duration_band", "spud_popularity_band",
    )
    .join(track_degrees, track_nodes.track_node_id == track_degrees.dst_track_node_id, "inner")
    .drop("dst_track_node_id")
    .join(spud_tracks.select("spotify_track_id", "duration_seconds"), "spotify_track_id", "left")
    .withColumn("log_playlist_degree", F.log1p("playlist_degree"))
    .withColumn("log_duration_seconds", F.log1p("duration_seconds"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

artist_degrees = track_artist_edges.groupBy("dst_artist_node_id").count().withColumnRenamed("count", "track_degree")
artist_analysis = (
    artist_nodes.select(
        "artist_node_id", "spotify_artist_id", "artist_name", "artist_metadata_available",
        "artist_followers", "artist_popularity_context", "follower_band", "genres",
    )
    .join(artist_degrees, artist_nodes.artist_node_id == artist_degrees.dst_artist_node_id, "inner")
    .drop("dst_artist_node_id")
    .withColumn("log_track_degree", F.log1p("track_degree"))
    .withColumn("log_followers", F.when(F.col("artist_followers").isNotNull(), F.log1p("artist_followers")))
    .withColumn("genre_count", F.size("genres").cast("double"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print("Analysis views created.")

## 3. Correlation matrices

Pearson matrices are calculated in Spark. Log transforms are used for strongly skewed counts so the relationship is not dominated by a few very large nodes.

For dissertation readability, every heatmap displays the complete symmetric matrix. The upper and lower triangles contain the same pairwise values because $\mathrm{corr}(A,B)=\mathrm{corr}(B,A)$, while the diagonal is always 1.00.

In [ ]:
def spark_correlation_matrix(dataframe, columns):
    numeric = dataframe.select(*[F.col(column).cast("double").alias(column) for column in columns]).na.drop()
    assembled = VectorAssembler(inputCols=columns, outputCol="_features", handleInvalid="skip").transform(numeric)
    matrix = Correlation.corr(assembled, "_features", "pearson").first()[0].toArray()
    return pd.DataFrame(matrix, index=columns, columns=columns)

def save_heatmap(matrix_pdf, title, filename, figsize):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        matrix_pdf, annot=True, fmt=".2f", cmap="vlag", center=0,
        vmin=-1, vmax=1, square=True, linewidths=0.4, cbar_kws={"label": "Pearson r"}, ax=ax,
    )
    ax.set_title(title, pad=12)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / filename, bbox_inches="tight")
    plt.show()

PLAYLIST_CORRELATION_COLUMNS = [
    "log_track_count", "artist_count", "log_duration_seconds",
    "mean_spud_popularity", "audio_coverage_ratio", "title_character_count",
]
playlist_corr_pdf = spark_correlation_matrix(playlist_analysis, PLAYLIST_CORRELATION_COLUMNS)
save_heatmap(
    playlist_corr_pdf, "Playlist-level numeric correlations",
    "playlist_correlation_matrix.png", (9, 7),
)
display(playlist_corr_pdf.round(4))

**Observed.** Playlist size and duration contain almost the same information ($r=0.978$), and larger playlists also contain more distinct artists ($r=0.789$). Mean SPUD popularity is largely independent of playlist size, while its moderate association with audio coverage ($r=0.385$) warns that the audio-matched subset is not a neutral sample of all playlists.

In [ ]:
TRACK_STRUCTURE_COLUMNS = [
    "spud_popularity", "log_duration_seconds", "log_playlist_degree", "audio_features_available",
]
track_structure_corr_pdf = spark_correlation_matrix(track_analysis, TRACK_STRUCTURE_COLUMNS)
save_heatmap(
    track_structure_corr_pdf, "Track metadata and graph-degree correlations",
    "track_structure_correlation_matrix.png", (7, 6),
)

catalog_audio = catalog_tracks.select(
    F.col("id").alias("spotify_track_id"),
    F.col("popularity").cast("double").alias("catalog_popularity"),
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "release_year",
)
matched_audio_analysis = track_analysis.where("audio_features_available = 1").join(
    catalog_audio, on="spotify_track_id", how="inner"
)
AUDIO_CORRELATION_COLUMNS = [
    "spud_popularity", "catalog_popularity", "log_playlist_degree",
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "release_year",
]
audio_corr_pdf = spark_correlation_matrix(matched_audio_analysis, AUDIO_CORRELATION_COLUMNS)
save_heatmap(
    audio_corr_pdf, f"Audio correlations on {matched_audio_analysis.count():,} matched playlist tracks",
    "matched_audio_correlation_matrix.png", (14, 12),
)
display(track_structure_corr_pdf.round(4))

**Observed.** Across all playlist tracks, SPUD popularity has only a weak positive association with playlist degree ($r=0.176$), although audio-matched tracks are somewhat more popular ($r=0.242$). In the 11,265-track matched subset, the strongest audio relationship is energy–loudness ($r=0.752$); energy–acousticness is strongly negative ($r=-0.711$). The two popularity scales agree reasonably well ($r=0.669$), but they are not interchangeable.

In [ ]:
ARTIST_CORRELATION_COLUMNS = [
    "log_track_degree", "log_followers", "artist_popularity_context", "genre_count",
]
artist_corr_pdf = spark_correlation_matrix(
    artist_analysis.where("artist_metadata_available = 1"), ARTIST_CORRELATION_COLUMNS
)
save_heatmap(
    artist_corr_pdf, "Artist metadata and connected-track correlations",
    "artist_correlation_matrix.png", (7, 6),
)
display(artist_corr_pdf.round(4))

**Observed.** Artist followers and catalog popularity are very strongly related ($r=0.889$). Artist connectivity also rises with followers ($r=0.553$), popularity ($r=0.480$), and genre count ($r=0.483$), so hub artists are not representative of the full artist catalog.

## 4. Playlist and metadata-coverage charts

In [ ]:
playlist_band_order = [
    "under_5_not_eligible", "small_5_9", "medium_10_24",
    "large_25_99", "very_large_100_plus",
]
playlist_bands_pdf = (
    playlist_nodes.groupBy("playlist_size_band").count().toPandas()
    .set_index("playlist_size_band").reindex(playlist_band_order).reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=playlist_bands_pdf, x="playlist_size_band", y="count", color=sns.color_palette("colorblind")[0], ax=axes[0])
axes[0].set_title("Playlists by size segment")
axes[0].set_xlabel("Playlist size segment")
axes[0].set_ylabel("Number of playlists")
axes[0].tick_params(axis="x", rotation=35)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%d", padding=2, fontsize=8)

playlist_histogram = playlist_analysis.select("track_count").rdd.flatMap(lambda row: row).histogram([1, 5, 10, 25, 50, 100, 200, 315])
bin_edges, bin_counts = playlist_histogram
labels = [f"{int(left)}-{int(right - 1)}" for left, right in zip(bin_edges[:-1], bin_edges[1:])]
axes[1].bar(labels, bin_counts, color=sns.color_palette("colorblind")[1])
axes[1].set_title("Playlist track-count distribution")
axes[1].set_xlabel("Tracks per playlist")
axes[1].set_ylabel("Number of playlists")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "playlist_size_distributions.png", bbox_inches="tight")
plt.show()
display(playlist_bands_pdf)

**Observed.** The connected graph is dominated by medium and large playlists: 6,545 contain 10–24 tracks and 7,075 contain 25–99. Only 420 connected playlists have fewer than five tracks, so 18,409 of 18,829 (97.8%) are eligible for seed/hidden-track evaluation. The long right tail still includes 3,711 playlists with at least 100 tracks.

In [ ]:
coverage_pdf = pd.DataFrame([
    {"entity": "Playlist tracks", "status": "Audio metadata available", "count": track_nodes.where("audio_features_available = 1").count()},
    {"entity": "Playlist tracks", "status": "Audio metadata missing", "count": track_nodes.where("audio_features_available = 0").count()},
    {"entity": "Artists", "status": "Catalog metadata available", "count": artist_nodes.where("artist_metadata_available = 1").count()},
    {"entity": "Artists", "status": "Catalog metadata missing", "count": artist_nodes.where("artist_metadata_available = 0").count()},
])
coverage_pdf["percentage_within_entity"] = coverage_pdf.groupby("entity")["count"].transform(lambda values: 100 * values / values.sum())
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=coverage_pdf, x="entity", y="percentage_within_entity", hue="status",
    palette="colorblind", ax=ax,
)
ax.set_title("Cross-dataset metadata coverage")
ax.set_xlabel("Graph entity")
ax.set_ylabel("Share of nodes (%)")
ax.legend(title="Join outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "metadata_coverage.png", bbox_inches="tight")
plt.show()
display(coverage_pdf.round(3))

**Observed.** Cross-dataset coverage is sparse: audio features are available for only 11,265 of 415,120 tracks (2.714%), and catalog metadata is available for 18,924 of 71,655 artists (26.410%). Collaborative graph signals must therefore remain the primary information source; content metadata is suitable as optional side information or for a restricted ablation.

## 5. Graph-degree, popularity, and genre charts

Logarithmic degree views reveal long-tail structure without allowing a few hubs to flatten the rest of the distribution.

In [ ]:
track_log_degree_pdf = (
    track_analysis.withColumn("log10_degree_bin", F.floor(F.log10("playlist_degree") * 10) / 10)
    .groupBy("log10_degree_bin").count().orderBy("log10_degree_bin").toPandas()
)
artist_log_degree_pdf = (
    artist_analysis.withColumn("log10_degree_bin", F.floor(F.log10("track_degree") * 10) / 10)
    .groupBy("log10_degree_bin").count().orderBy("log10_degree_bin").toPandas()
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(track_log_degree_pdf["log10_degree_bin"], track_log_degree_pdf["count"], marker="o", markersize=3)
axes[0].set_yscale("log")
axes[0].set_title("Track popularity in the playlist graph")
axes[0].set_xlabel("log10(number of connected playlists)")
axes[0].set_ylabel("Number of tracks (log scale)")
axes[1].plot(artist_log_degree_pdf["log10_degree_bin"], artist_log_degree_pdf["count"], marker="o", markersize=3, color=sns.color_palette("colorblind")[2])
axes[1].set_yscale("log")
axes[1].set_title("Artist connectivity in the track graph")
axes[1].set_xlabel("log10(number of connected tracks)")
axes[1].set_ylabel("Number of artists (log scale)")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "graph_degree_distributions.png", bbox_inches="tight")
plt.show()

**Observed.** Both node types have long-tailed degree distributions. A typical track occurs in one playlist (50th percentile), while the 90th and 99th percentiles are 5 and 20 playlists. Artist track degrees are 2, 13, and 59 at the same percentiles. Evaluation should therefore separate head and tail items rather than reporting only a single average accuracy score.

In [ ]:
track_sample_pdf = (
    track_analysis.select("spud_popularity", "playlist_degree")
    .sample(False, 0.15, seed=42).limit(50_000).toPandas()
)
fig, ax = plt.subplots(figsize=(9, 6))
hexbin = ax.hexbin(
    track_sample_pdf["spud_popularity"], np.log1p(track_sample_pdf["playlist_degree"]),
    gridsize=45, mincnt=1, bins="log", cmap="viridis",
)
fig.colorbar(hexbin, ax=ax, label="log10(sampled track count per hexagon)")
ax.set_title("SPUD popularity versus playlist connectivity")
ax.set_xlabel("SPUD popularity (0-1)")
ax.set_ylabel("log(1 + connected playlists)")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "track_popularity_vs_degree.png", bbox_inches="tight")
plt.show()

**Observed.** Higher SPUD popularity is associated with greater playlist connectivity, but the relationship is weak ($r=0.176$ after log-transforming degree) and the dense cloud remains broad. Popularity alone cannot explain playlist membership, although it can still create exposure bias in ranking.

In [ ]:
top_genres_pdf = (
    artist_nodes.select(F.explode("genres").alias("genre"))
    .where(F.length(F.trim("genre")) > 0)
    .groupBy("genre").count().orderBy(F.desc("count"), "genre").limit(20).toPandas()
)
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top_genres_pdf.sort_values("count"), x="count", y="genre", color=sns.color_palette("colorblind")[4], ax=ax)
ax.set_title("Most represented genres among metadata-matched artists")
ax.set_xlabel("Number of connected artists")
ax.set_ylabel("Genre")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "top_artist_genres.png", bbox_inches="tight")
plt.show()
display(top_genres_pdf)

**Observed.** Rock is the most represented genre among metadata-matched artists (511), followed by dance pop (417) and country rock (393). The head contains many overlapping rock and pop subgenres, so these counts describe catalog tags rather than mutually exclusive artist classes. They also apply only to the 26.4% of connected artists with matched metadata.

## 6. Save correlation tables and print computed observations

In [ ]:
playlist_corr_pdf.to_csv(REPORTS_ROOT / "playlist_correlation_matrix.csv")
track_structure_corr_pdf.to_csv(REPORTS_ROOT / "track_structure_correlation_matrix.csv")
audio_corr_pdf.to_csv(REPORTS_ROOT / "matched_audio_correlation_matrix.csv")
artist_corr_pdf.to_csv(REPORTS_ROOT / "artist_correlation_matrix.csv")

def strongest_pair(matrix_pdf):
    pairs = []
    for left_index, left_name in enumerate(matrix_pdf.columns):
        for right_index in range(left_index + 1, len(matrix_pdf.columns)):
            right_name = matrix_pdf.columns[right_index]
            value = float(matrix_pdf.iloc[left_index, right_index])
            if not math.isnan(value):
                pairs.append((abs(value), value, left_name, right_name))
    return max(pairs)

playlist_top = strongest_pair(playlist_corr_pdf)
audio_top = strongest_pair(audio_corr_pdf)
artist_top = strongest_pair(artist_corr_pdf)
playlist_median = playlist_analysis.approxQuantile("track_count", [0.5], 0.001)[0]
track_degree_quantiles = track_analysis.approxQuantile("playlist_degree", [0.5, 0.9, 0.99], 0.001)
artist_degree_quantiles = artist_analysis.approxQuantile("track_degree", [0.5, 0.9, 0.99], 0.001)
eligible_count = playlist_nodes.where("eligible_for_link_prediction = 1").count()
audio_coverage_pct = 100 * track_nodes.where("audio_features_available = 1").count() / track_nodes.count()
artist_coverage_pct = 100 * artist_nodes.where("artist_metadata_available = 1").count() / artist_nodes.count()

print("DATA ANALYSIS OBSERVATIONS")
print("==========================")
print(f"1. {eligible_count:,} of {analysis_counts['playlist_nodes']:,} connected playlists have at least five tracks and are evaluation-eligible.")
print(f"2. The median playlist contains {playlist_median:.0f} tracks; playlist lengths are strongly right-skewed.")
print(f"3. The strongest playlist-level numeric pair is {playlist_top[2]} vs {playlist_top[3]} (r={playlist_top[1]:.3f}).")
print(f"4. Track playlist-degree quantiles (50th/90th/99th percentiles) are {track_degree_quantiles}. This confirms a long popularity tail.")
print(f"5. Artist track-degree quantiles (50th/90th/99th percentiles) are {artist_degree_quantiles}. A small set of artists acts as graph hubs.")
print(f"6. The strongest matched-audio pair is {audio_top[2]} vs {audio_top[3]} (r={audio_top[1]:.3f}).")
print(f"7. The strongest artist-level pair is {artist_top[2]} vs {artist_top[3]} (r={artist_top[1]:.3f}).")
print(f"8. Audio metadata covers only {audio_coverage_pct:.3f}% of playlist tracks; artist catalog metadata covers {artist_coverage_pct:.3f}% of connected artists.")
print("9. The low cross-dataset coverage supports using a collaborative GNN as the primary model and treating audio/artist attributes as optional side information.")
print("10. Popularity and full-graph degree relationships indicate a popularity-bias risk; evaluation should report catalog coverage and performance by playlist-size/popularity segment.")
print("11. Correlations describe the available graph but are not predictive evaluation. Recommendation quality must be measured on hidden playlist-track links.")
print(f"12. Saved {len(list(FIGURE_ROOT.glob('*.png')))} analysis figures to {FIGURE_ROOT.relative_to(PROJECT_ROOT)}.")